# Markov Models for Text Generation

A First Generative Model From Scratch

In this set of notes, we’ll take our first peak “under the hood” into a model for text generation which is simple enough for us to code from scratch and train almost instantly. This model is called a *Markov model*.

We’ll also introduce *tokenization*, which prepares text data for use in machine learning models by breaking it down into smaller units.

## Tokenization

Here’s a simple piece of text for us to consider:

In [ ]:
text = """
Congratulations! 
Today is your day! 
You're off to Great Places! 
You're off and away! 
You have brains in your head.
You have feet in your shoes.
You can steer yourself
any direction you choose.
You're on your own. And you know what you know.
And YOU are the one who'll decide where to go.
"""

text = """
Congratulations!
Today is your day.
You're off to Great Places!
You're off and away!

You have brains in your head.
You have feet in your shoes.
You can steer yourself
any direction you choose.
You're on your own. And you know what you know.
And YOU are the guy who'll decide where to go.

You'll look up and down streets. Look 'em over with care.
About some you will say, "I don't choose to go there."
With your head full of brains and your shoes full of feet,
you're too smart to go down any not-so-good street.

And you may not find any
you'll want to go down.
In that case, of course,
you'll head straight out of town.

It's opener there
in the wide open air.

Out there things can happen
and frequently do
to people as brainy
and footsy as you.

And then things start to happen,
don't worry. Don't stew.
Just go right along.
You'll start happening too.

OH!
THE PLACES YOU'LL GO!

You'll be on y our way up!
You'll be seeing great sights!
You'll join the high fliers
who soar to high heights.

You won't lag behind, because you'll have the speed.
You'll pass the whole gang and you'll soon take the lead.
Wherever you fly, you'll be best of the best.
Wherever you go, you will top all the rest.

Except when you don't.
Because, sometimes, you won't.

I'm sorry to say so
but, sadly, it's true
that Bang-ups
and Hang-ups
can happen to you.

You can get all hung up
in a prickle-ly perch.
And your gang will fly on.
You'll be left in a Lurch.

You'll come down from the Lurch
with an unpleasant bump.
And the chances are, then,
that you'll be in a Slump.

And when you're in a Slump,
you're not in for much fun.
Un-slumping yourself
is not easily done.

You will come to a place where the streets are not marked.
Some windows are lighted. But mostly they're darked.
A place you could sprain both your elbow and chin!
Do you dare to stay out? Do you dare to go in?
How much can you lose? How much can you win?

And IF you go in, should you turn left or right...
or right-and-three-quarters? Or, maybe, not quite?
Or go around back and sneak in from behind?
Simple it's not, I'm afraid you will find,
for a mind-maker-upper to make up his mind.

You can get so confused
that you'll start in to race
down long wiggled roads at a break-necking pace
and grind on for miles cross weirdish wild space,
headed, I fear, toward a most useless place.
The Waiting Place...

...for people just waiting.
Waiting for a train to go
or a bus to come, or a plane to go
or the mail to come, or the rain to go
or the phone to ring, or the snow to snow
or the waiting around for a Yes or No
or waiting for their hair to grow.
Everyone is just waiting.

Waiting for the fish to bite
or waiting for the wind to fly a kite
or waiting around for Friday night
or waiting, perhaps, for their Uncle Jake
or a pot to boil, or a Better Break
or a string of pearls, or a pair of pants
or a wig with curls, or Another Chance.
Everyone is just waiting.

NO!
That's not for you!

Somehow you'll escape
all that waiting and staying
You'll find the bright places
where Boom Bands are playing.

With banner flip-flapping,
once more you'll ride high!
Ready for anything under the sky.
Ready because you're that kind of a guy!

Oh, the places you'll go! There is fun to be done!
There are points to be scored. There are games to be won.
And the magical things you can do with that ball
will make you the winning-est winner of all.
Fame! You'll be as famous as famous can be,
with the whole wide world watching you win on TV.

Except when they don't
Because, sometimes they won't.

I'm afraid that some times
you'll play lonely games too.
Games you can't win
'cause you'll play against you.

All Alone!
Whether you like it or not,
Alone will be something
you'll be quite a lot.

And when you're alone, there's a very good chance
you'll meet things that scare you right out of your pants.
There are some, down the road between hither and yon,
that can scare you so much you won't want to go on.

But on you will go
though the weather be foul.
On you will go
though your enemies prowl.
On you will go
though the Hakken-Kraks howl.
Onward up many
a frightening creek,
though your arms may get sore
and your sneakers may leak.

On and on you will hike,
And I know you'll hike far
and face up to your problems
whatever they are.

You'll get mixed up, of course,
as you already know.
You'll get mixed up
with many strange birds as you go.
So be sure when you step.
Step with care and great tact
and remember that Life's
a Great Balancing Act.
Just never foget to be dexterous and deft.
And never mix up your right foot with your left.

And will you succeed?
Yes! You will, indeed!
(98 and 3/4 percent guaranteed.)

KID, YOU'LL MOVE MOUNTAINS!

So...
be your name Buxbaum or Bixby or Bray
or Mordecai Ali Van Allen O'Shea,
You're off the Great Places!
Today is your day!
Your mountain is waiting.
So...get on your way!
"""

We’d like to build a model that can generate text which is “similar to this.” The idea of “similarity” here requires specification, and the task of operationalizing the idea of “similarity” is one of the core parts of language modeling. All modern approaches to language modeling begin by viewing a given text as a sequence of *tokens*, which are “parts of language.” One way to tokenize this text is to split it into individual characters:

In [ ]:
char_tokens = list(text)
print(char_tokens[0:20])

Another reasonable choice is to split the text into words:

In [ ]:
word_tokens = text.split() # split on the space \s character
print(word_tokens[0:20])

Note that this approach has some limitations:

-   Punctuation is attached to words (e.g., “Congratulations!” is a single token)
-   Capitalization is preserved (e.g., “You” and “you” are different tokens)

Modern tokenizers implement complex pipelines for breaking text into tokens. Here’s an example:

In [ ]:
from collections import defaultdict

def get_most_common_pair(tokens): 
    pair_counts = defaultdict(int)
    for i in range(len(tokens) - 1): 
        pair = (tokens[i], tokens[i+1])
        pair_counts[pair] += 1
    most_common_pair = max(pair_counts, key=pair_counts.get)
    return most_common_pair, pair_counts[most_common_pair]

class BPETokenizer: 

    def __init__(self): 
        self.encoding_map = {}
        self.decoding_map = {}

    def training_step(self, tokens): 
        most_common_pair, count = get_most_common_pair(tokens)
        if count < 2: 
            return tokens  # stop if no pair occurs more than once
        new_token = ''.join(most_common_pair)
        # update encoder and decoder
        self.encoding_map[most_common_pair] = new_token
        self.decoding_map[new_token] = most_common_pair
    
        new_tokens = []
        i = 0
        while i < len(tokens): 
            if i < len(tokens) - 1 and (tokens[i], tokens[i+1]) == most_common_pair: 
                new_tokens.append(new_token)
                i += 2
            else: 
                new_tokens.append(tokens[i])
                i += 1
        return new_tokens
    
    def train(self, tokens, max_steps, verbose = False):
        new_tokens = tokens 
        for _ in range(max_steps): 
            if verbose:
                print(f"Step {_}: vocab size = {len(self.encoding_map) + len(set(tokens))}")
            new_tokens = self.training_step(new_tokens)
            if len(new_tokens) == len(tokens): 
                break

        self.vocab_size = len(self.encoding_map)

    def encoding_step(self, tokens): 
        encoded_tokens = []
        i = 0
        while i < len(tokens): 
            if i < len(tokens) - 1 and (tokens[i], tokens[i+1]) in self.encoding_map: 
                encoded_tokens.append(self.encoding_map[(tokens[i], tokens[i+1])])
                i += 2
            else: 
                encoded_tokens.append(tokens[i])
                i += 1

        return encoded_tokens

    def encode(self, tokens): 
        encoded_tokens = tokens
        while True: 
            new_tokens = self.encoding_step(encoded_tokens)
            if len(new_tokens) == len(encoded_tokens): 
                break
            encoded_tokens = new_tokens
        return encoded_tokens

    def decoding_step(self, tokens): 
        decoded_tokens = []
        for token in tokens: 
            if token in self.decoding_map: 
                decoded_tokens.extend(self.decoding_map[token])
            else: 
                decoded_tokens.append(token)
        return decoded_tokens

    def decode(self, tokens): 
        decoded_tokens = tokens
        while True: 
            new_tokens = self.decoding_step(decoded_tokens)
            if len(new_tokens) == len(decoded_tokens): 
                break
            decoded_tokens = new_tokens
        return decoded_tokens

In [ ]:
bpe_tokenizer = BPETokenizer()
tokens = list(text)

bpe_tokenizer.train(tokens, max_steps=1000, verbose = True)

bpe_tokenizer.vocab

encoded_tokens = bpe_tokenizer.encode(tokens)
decoded_tokens = bpe_tokenizer.decode(encoded_tokens)

In [ ]:
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
# from tokenizers.pre_tokenizers import Whitespace

tokenizer = Tokenizer(BPE())
trainer = BpeTrainer(min_frequency=10)

tokenizer.train_from_iterator([text], trainer, )

encoded = tokenizer.encode(text).ids

vocab = tokenizer.get_vocab()
print(vocab)

In [ ]:
from collections import defaultdict
import random

def sample_from_counter(counter):
    total = sum(counter.values())
    probabilities = [count / total for count in counter.values()]
    return random.choices(list(counter.keys()), probabilities)[0]

class MarkovModel:
    def __init__(self, n):
        self.n = n
        self.transitions = defaultdict(lambda: defaultdict(int))
        self.context_counts = defaultdict(int)

    def train(self, tokens):

        for i in range(len(tokens) - self.n):
            if self.n == 0: 
                context = tuple()
            else:
                context = tuple(tokens[i:i + self.n])
            next_token = tokens[i + self.n]
            self.transitions[context][next_token] += 1
            self.context_counts[context] += 1
                
    def generate(self, start_context = None, length = 100):

        if start_context is None:
            start_context = sample_from_counter(self.context_counts)
        
        if self.n == 0:
            context = tuple()

        context = tuple(start_context)
        result = list(context)
        for _ in range(length):
            if context not in self.transitions:
                break
            next_tokens = list(self.transitions[context].keys())
            weights = list(self.transitions[context].values())
            next_token = sample_from_counter(self.transitions[context])
            result.append(next_token)
            if self.n == 0:
                context = tuple()
            else: 
                context = tuple(result[-self.n:])
        return result

In [ ]:
n = 1
model = MarkovModel(n)
model.train(encoded)

In [ ]:
import re 

def format_seuss(text):
    output = re.sub(r'(\s)(?=[A-z])', r'', text)
    output = re.sub(r'(\s)(?=\')', r'', output)
    output = re.sub(r'(\s)(?=[!.?,)()])', r'', output)
    return output

generated_ids = model.generate(length=500)
generated_tokens = tokenizer.decode(generated_ids)

print(format_seuss(generated_tokens))



# print(generated_tokens)